In [ ]:
import os

# ===== 镜像配置（放在最前面）=====
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 代理（根据你的网络环境）
# os.environ['HTTP_PROXY'] = 'http://192.168.1.100:10808'
# os.environ['HTTPS_PROXY'] = 'http://192.168.1.100:10808'

import torch
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
from diffusers import StableDiffusionInpaintPipeline

DEVICE = "cuda"

def load_sam():
    print("[INFO] Loading SAM...")
    sam = sam_model_registry["vit_h"](checkpoint="./cache/sam_vit_h_4b8939.pth")
    sam.to(DEVICE)
    return SamAutomaticMaskGenerator(sam)

def load_inpaint():
    print("[INFO] Loading Inpainting model...")
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        "runwayml/stable-diffusion-inpainting",
        cache_dir="./cache",
        torch_dtype=torch.float16
    ).to(DEVICE)
    pipe.enable_attention_slicing()
    return pipe

# 初始化模型
sam_generator = load_sam()
inpaint_pipe = load_inpaint()

In [ ]:
def create_test_graffiti_ref(save_path: str = "./output/ref_graffiti.png"):
    """生成一张测试用的涂鸦参考图（为了方便你跑通流程）"""
    print("[INFO] Generating a test graffiti reference image...")
    img = Image.new('RGBA', (512, 512), (255, 255, 255, 0)) # 透明背景
    draw = ImageDraw.Draw(img)
    
    # 画一些随机的涂鸦线条和圆
    colors = ['#FF5733', '#33FF57', '#3357FF', '#F033FF']
    for _ in range(5):
        color = random.choice(colors)
        # 随机画粗线条
        x1, y1 = random.randint(50, 200), random.randint(50, 450)
        x2, y2 = random.randint(300, 460), random.randint(50, 450)
        draw.line([(x1, y1), (x2, y2)], fill=color, width=15)
        # 随机画圆
        draw.ellipse([x1-40, y1-40, x1+40, y1+40], fill=color, outline='black', width=5)
        
    # 写个字
    try:
        font = ImageFont.truetype("arial.ttf", 80)
    except:
        font = ImageFont.load_default()
    draw.text((80, 200), "COOL", fill='black', font=font, stroke_width=3, stroke_fill='white')
    
    img.save(save_path)
    print(f"[INFO] Test reference saved to {save_path}")
    return np.array(img)

def get_largest_plane_mask(image: np.ndarray, sam_gen) -> np.ndarray:
    masks = sam_gen.generate(image)
    if not masks: raise ValueError("未检测到任何区域！")
    masks = sorted(masks, key=lambda x: x['area'], reverse=True)
    return (masks[0]['segmentation'] * 255).astype(np.uint8)

def sample_patch_on_plane(plane_mask: np.ndarray, img_shape: tuple, patch_config: dict):
    """采样Patch，并返回Mask和带有轻微透视倾斜的四个角点"""
    H, W = img_shape[:2]
    min_coverage = patch_config.get("min_coverage", 0.80)
    max_attempts = patch_config.get("max_attempts", 100)
    
    mode = patch_config.get("mode", "ratio")
    base_size = int(min(H, W) * patch_config.get("value", 0.15)) if mode == "ratio" else patch_config.get("value", 200)
    min_size, max_size = int(base_size * 0.8), int(base_size * 1.2)
    
    for attempt in range(max_attempts):
        w = random.randint(min_size, max_size)
        h = int(w * random.uniform(0.8, 1.25))
        h = max(min_size, min(h, max_size))
        
        cx = random.randint(w // 2, W - w // 2)
        cy = random.randint(h // 2, H - h // 2)
        x1, x2 = cx - w // 2, cx + w // 2
        y1, y2 = cy - h // 2, cy + h // 2
        
        if x1 < 0 or y1 < 0 or x2 > W or y2 > H: continue
        
        crop_mask = plane_mask[y1:y2, x1:x2]
        if np.sum(crop_mask == 255) / (w * h) >= min_coverage:
            print(f"[INFO] Found valid patch at attempt {attempt+1}.")
            patch_mask = np.zeros((H, W), dtype=np.uint8)
            patch_mask[y1:y2, x1:x2] = 255
            
            # 计算带有随机倾斜的角点（模拟手绘透视感）
            jitter = max(5, int(base_size * 0.04)) 
            pts = np.array([
                [x1 + random.randint(-jitter, jitter), y1 + random.randint(-jitter, jitter)],
                [x2 + random.randint(-jitter, jitter), y1 + random.randint(-jitter, jitter)],
                [x2 + random.randint(-jitter, jitter), y2 + random.randint(-jitter, jitter)],
                [x1 + random.randint(-jitter, jitter), y2 + random.randint(-jitter, jitter)]
            ], dtype=np.float32)
            return patch_mask, pts
            
    raise ValueError("采样失败！")

# def blend_ref_as_draft(image_cv: np.ndarray, pts: np.ndarray, ref_img_np: np.ndarray, alpha: float = 0.6):
#     """将参考涂鸦图做透视变换，并作为半透明底稿融合到墙面"""
#     H, W = image_cv.shape[:2]
#     h, w = ref_img_np.shape[:2]
    
#     # 1. 参考图透视变形贴合墙面
#     src_pts = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32)
#     M = cv2.getPerspectiveTransform(src_pts, pts)
#     warped_ref = cv2.warpPerspective(ref_img_np, M, (W, H))
    
#     # 2. 提取区域Mask
#     mask = np.zeros((H, W), dtype=np.uint8)
#     cv2.fillConvexPoly(mask, pts.astype(np.int32), 255)
#     mask_3ch = cv2.merge([mask, mask, mask])
    
#     # 3. Alpha混合 (保留部分原图墙面纹理 + 叠加参考图结构)
#     blended = np.where(mask_3ch == 255, 
#                        cv2.addWeighted(warped_ref, alpha, image_cv, 1 - alpha, 0), 
#                        image_cv)
#     return blended

def blend_ref_as_draft(image_cv: np.ndarray, pts: np.ndarray, ref_img_np: np.ndarray, alpha: float = 0.6):
    """将参考涂鸦图做透视变换，并作为半透明底稿融合到墙面"""
    H, W = image_cv.shape[:2]
    h, w = ref_img_np.shape[:2]
    
    # 1. 处理参考图：RGBA转RGB
    if ref_img_np.shape[2] == 4:
        ref_rgb = ref_img_np[:, :, :3]
    else:
        ref_rgb = ref_img_np
    
    # 2. 透视变换
    src_pts = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(src_pts, pts)
    warped_ref = cv2.warpPerspective(ref_rgb, M, (W, H))
    
    # 3. 确保数据类型一致
    if warped_ref.dtype != image_cv.dtype:
        warped_ref = warped_ref.astype(image_cv.dtype)
    
    # 4. 创建mask
    mask = np.zeros((H, W), dtype=np.uint8)
    cv2.fillConvexPoly(mask, pts.astype(np.int32), 255)
    
    # 5. 创建结果图像
    blended = image_cv.copy()
    
    # 6. 只对mask区域进行混合
    y_inds, x_inds = np.where(mask == 255)
    
    for y, x in zip(y_inds, x_inds):
        blended[y, x] = (warped_ref[y, x].astype(np.float32) * alpha + 
                         image_cv[y, x].astype(np.float32) * (1 - alpha)).astype(image_cv.dtype)
    
    return blended




In [ ]:
def generate_graffiti_from_ref(
    image_path: str, 
    ref_image_path: str,
    output_dir: str = "./output",
    patch_config: dict = None
):
    import os
    os.makedirs(output_dir, exist_ok=True)
    if patch_config is None:
        patch_config = {"mode": "ratio", "value": 0.20}

    # 1. 读取输入图
    image_cv = cv2.imread(image_path)
    image_cv = cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB)
    
    # 2. 读取参考图 (支持带透明通道PNG)
    ref_img_pil = Image.open(ref_image_path).convert("RGBA")
    ref_img_np = np.array(ref_img_pil)
    
    # 3. SAM 检测与采样
    plane_mask = get_largest_plane_mask(image_cv, sam_generator)
    cv2.imwrite(f"{output_dir}/plane_mask.png", plane_mask)
    
    patch_mask, pts = sample_patch_on_plane(plane_mask, image_cv.shape, patch_config)
    cv2.imwrite(f"{output_dir}/patch_mask.png", patch_mask)
    
    # 4. 融合底稿：将参考图作为“半成品涂鸦”贴到墙上
    # alpha越高，参考图轮廓越清晰；alpha越低，SD自由度越高
    init_image_with_draft = blend_ref_as_draft(image_cv, pts, ref_img_np, alpha=0.6)
    init_image_bgr = cv2.cvtColor(init_image_with_draft, cv2.COLOR_RGB2BGR)
    cv2.imwrite(f"{output_dir}/init_draft.jpg", init_image_bgr)
    
    # 5. 准备 SD 输入
    init_image_pil = Image.fromarray(init_image_with_draft).convert("RGB")
    mask_image_pil = Image.fromarray(patch_mask).convert("RGB")
    
    # 6. 执行 Inpainting (Prompt 引导它转化为真实材质)
    prompt = "realistic graffiti art on a brick wall, spray paint texture, drips, street art, highly detailed"
    negative_prompt = "clean, flat, digital art, illustration, perfect lines, sticker, paper"
    
    print("[INFO] Running Inpainting based on reference draft...")
    result = inpaint_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=init_image_pil,
        mask_image=mask_image_pil,
        num_inference_steps=50
    ).images[0]
    
    result_path = f"{output_dir}/result_graffiti.jpg"
    result.save(result_path)
    
    return image_cv, plane_mask, ref_img_np, init_image_with_draft, np.array(result)

In [ ]:
# ================= 配置参数 =================
INPUT_IMAGE = "./input.jpg"  
# REF_IMAGE = "./my_custom_graffiti.png" # 实际使用时替换为你的涂鸦图
PATCH_CONFIG = {"mode": "ratio", "value": 0.30} 
# ==========================================

import os
os.makedirs("./output", exist_ok=True)

# 0. 如果没有涂鸦参考图，先自动生成一张用于测试
REF_IMAGE = "./output/ref_graffiti.png"
if not os.path.exists(REF_IMAGE):
    ref_img_np = create_test_graffiti_ref(REF_IMAGE)
else:
    ref_img_np = np.array(Image.open(REF_IMAGE).convert("RGBA"))

# 执行生成
img_orig, m_plane, ref_np, draft_img, img_result = generate_graffiti_from_ref(
    image_path=INPUT_IMAGE,
    ref_image_path=REF_IMAGE,
    patch_config=PATCH_CONFIG
)

# 可视化全过程
plt.figure(figsize=(20, 4))

plt.subplot(1, 5, 1)
plt.title("1. Input Wall")
plt.imshow(img_orig)
plt.axis('off')

plt.subplot(1, 5, 2)
plt.title("2. SAM Plane Mask")
plt.imshow(m_plane, cmap='gray')
plt.axis('off')

plt.subplot(1, 5, 3)
plt.title("3. Reference Input\n(Your Graffiti)")
# 显示参考图，用灰格子背景表示透明度
plt.imshow(ref_np)
plt.axis('off')

plt.subplot(1, 5, 4)
plt.title("4. Draft (Sent to SD)\n[Ref + Wall Blended]")
plt.imshow(draft_img)
plt.axis('off')

plt.subplot(1, 5, 5)
plt.title("5. Final Inpainting Result\n[Realistic Texture]")
plt.imshow(img_result)
plt.axis('off')

plt.tight_layout()
plt.show()